<a href="https://colab.research.google.com/github/bah862696-coder/DI-Bootcamp/blob/master/ExerciceXP_week6_Day4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import time
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import copy

# Question 1: LoRALayer implementation
class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev = 1 / torch.sqrt(torch.tensor(rank).float())
        # Matrice A initialisée avec Gaussienne
        self.A = nn.Parameter(torch.randn(in_dim, rank) * std_dev)
        # Matrice B initialisée à zéro pour que l'ajustement initial soit nul
        self.B = nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha = alpha

    def forward(self, x):
        # Transformation LoRA : x * A * B
        return self.alpha * (x @ self.A @ self.B)

In [2]:
# Question 2 & 3: LinearWithLoRA and Testing
class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # Somme du poids original et de l'adaptation LoRA
        return self.linear(x) + self.lora(x)

# Test initial
torch.manual_seed(123)
x = torch.randn((1, 10))
layer = nn.Linear(10, 5)
print('Original output:', layer(x))

layer_lora_1 = LinearWithLoRA(layer, rank=4, alpha=8)
print('LoRA output (initial):', layer_lora_1(x))

Original output: tensor([[ 0.4257,  0.0299, -0.1865, -0.3084,  0.4543]],
       grad_fn=<AddmmBackward0>)
LoRA output (initial): tensor([[ 0.4257,  0.0299, -0.1865, -0.3084,  0.4543]], grad_fn=<AddBackward0>)


In [3]:
# Question 4: Merged LoRA Layer for Efficiency
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear = linear
        self.lora = LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        # Fusion mathématique : W_total = W + alpha * (A @ B).T
        lora_weights = (self.lora.A @ self.lora.B)
        combined_weight = self.linear.weight + self.lora.alpha * lora_weights.T
        return F.linear(x, combined_weight, self.linear.bias)

layer_lora_2 = LinearWithLoRAMerged(layer, rank=4, alpha=8)
print('Merged LoRA output:', layer_lora_2(x))

Merged LoRA output: tensor([[ 0.4257,  0.0299, -0.1865, -0.3084,  0.4543]],
       grad_fn=<AddmmBackward0>)


In [4]:
# Question 5: Multilayer Perceptron with LoRA
class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(num_features, num_hidden_1),
            nn.ReLU(),
            nn.Linear(num_hidden_1, num_hidden_2),
            nn.ReLU(),
            nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        return self.layers(x)

# Hyperparameters
num_features = 28*28
num_hidden_1 = 128
num_hidden_2 = 64
num_classes = 10
learning_rate = 0.001
num_epochs = 2
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = MultilayerPerceptron(num_features, num_hidden_1, num_hidden_2, num_classes)
model.to(DEVICE)

# Data Loading
BATCH_SIZE = 64
train_dataset = datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)
test_dataset = datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader = DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

100%|██████████| 9.91M/9.91M [00:00<00:00, 19.5MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 480kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.50MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 14.0MB/s]


In [5]:
# Training and Evaluation Functions
def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples = 0, 0
    with torch.no_grad():
        for features, targets in data_loader:
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            _, predicted_labels = torch.max(logits, 1)
            num_examples += targets.size(0)
            correct_pred += (predicted_labels == targets).sum()
        return correct_pred.float() / num_examples * 100

def train(num_epochs, model, optimizer, train_loader, device):
    start_time = time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features = features.to(device)
            targets = targets.to(device)
            logits = model(features)
            loss = F.cross_entropy(logits, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            if not batch_idx % 400:
                print(f'Epoch: {epoch+1:03d}/{num_epochs:03d} | Batch {batch_idx:03d}/{len(train_loader):03d} | Loss: {loss:.4f}')
    print(f'Total Training Time: {(time.time() - start_time)/60:.2f} min')

In [6]:
# Applying LoRA and Freezing (Question 6)
model_lora = copy.deepcopy(model)

# Replace layers
model_lora.layers[1] = LinearWithLoRAMerged(model_lora.layers[1], rank=4, alpha=8)
model_lora.layers[3] = LinearWithLoRAMerged(model_lora.layers[3], rank=4, alpha=8)
model_lora.layers[5] = LinearWithLoRAMerged(model_lora.layers[5], rank=4, alpha=8)

def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad = False
        else:
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
model_lora.to(DEVICE)

print("Trainable parameters status:")
for name, param in model_lora.named_parameters():
    print(f'{name}: {param.requires_grad}')

# Train LoRA model
optimizer_lora = torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Final LoRA Test accuracy: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

Trainable parameters status:
layers.1.linear.weight: False
layers.1.linear.bias: False
layers.1.lora.A: True
layers.1.lora.B: True
layers.3.linear.weight: False
layers.3.linear.bias: False
layers.3.lora.A: True
layers.3.lora.B: True
layers.5.linear.weight: False
layers.5.linear.bias: False
layers.5.lora.A: True
layers.5.lora.B: True
Epoch: 001/002 | Batch 000/938 | Loss: 2.3131
Epoch: 001/002 | Batch 400/938 | Loss: 1.0217
Epoch: 001/002 | Batch 800/938 | Loss: 0.5312
Epoch: 002/002 | Batch 000/938 | Loss: 0.5128
Epoch: 002/002 | Batch 400/938 | Loss: 0.5607
Epoch: 002/002 | Batch 800/938 | Loss: 0.6788
Total Training Time: 0.30 min
Final LoRA Test accuracy: 83.32%
